
This notebook demonstrates the process of registering (aligning) two 3D point cloud maps:

1. **Source Map**: The map generated by KISS-ICP (`Kiss_SLAM Generated Map`)

- **Downsampled Map**: We followed the highlights of Supervisor(`Faezeh Sadat Mortazavi`) to downsample the original SLAM map at 10cm using `CloudCompare Software`.

2. **Target Map**: The reference labeled map (`reference-labled_map.pcd`)

Map registration is a crucial step for tasks like change detection, as it ensures that both maps are in the same coordinate system, allowing for accurate comparison between them.

The workflow consists of the following steps:
1. **Loading Point Clouds**: Loading both the source and target point clouds
2. **Preprocessing**: Downsampling and filtering the point clouds to improve registration efficiency
3. **Initial Alignment**: Using feature-based methods for coarse alignment
4. **Fine Registration**: Refining the alignment using ICP (Iterative Closest Point) algorithm
5. **Evaluation**: Assessing the quality of the registration
6. **Visualization**: Visualizing the results before and after registration
7. **Saving Results**: Saving the aligned map for future use

## 1. Import Libraries

First, we import the necessary libraries for point cloud processing. We'll primarily use Open3D, which is a powerful library for 3D data processing.

In [1]:
import numpy as np
import open3d as o3d
import copy
import os
import time
import matplotlib.pyplot as plt

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## 2. Load Point Clouds

Next, we load both the source point cloud (Slam_Map.ply) and the target point cloud (reference-labled_map.pcd). We'll also display some basic information about each point cloud.

In [2]:
# Define file paths: Using original point cloud as per supervisor's recommendation for accurate registration.

# The original map generated by KISS-SLAM
#source_file = "Kiss_slam_group/results/final_map_from_precomputed_poses.ply"
source_file = "SLAM_New_Map_Cropped_Input.ply"
# The reference labeled map
target_file = "reference-labled_map.pcd"

# Check if files exists.
if not os.path.exists(source_file):
    raise FileNotFoundError(f"Source file not found: {source_file}")
if not os.path.exists(target_file):
    raise FileNotFoundError(f"Target file not found: {target_file}")

print(f"Loading source point cloud from: {source_file}")
source = o3d.io.read_point_cloud(source_file)

print(f"Loading target point cloud from: {target_file}")
target = o3d.io.read_point_cloud(target_file)

# Print information about the point clouds
print("\nSource point cloud:")
print(f"Number of points: {len(source.points)}")
print(f"Has colors: {source.has_colors()}")
print(f"Has normals: {source.has_normals()}")

print("\nTarget point cloud:")
print(f"Number of points: {len(target.points)}")
print(f"Has colors: {target.has_colors()}")
print(f"Has normals: {target.has_normals()}")

# Create a coordinate frame for reference
coordinate_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=5.0, origin=[0, 0, 0])

Loading source point cloud from: SLAM_New_Map_Cropped_Input.ply
Loading target point cloud from: reference-labled_map.pcd

Source point cloud:
Number of points: 7182404
Has colors: False
Has normals: True

Target point cloud:
Number of points: 20451227
Has colors: False
Has normals: False


## 4. Preprocess Point Clouds

To improve the efficiency and accuracy of registration, we'll preprocess the point clouds by:
1. Downsampling to reduce the number of points
2. Estimating normals (if not already present)
3. Removing outliers to improve registration quality

In [3]:
def preprocess_point_cloud(pcd, voxel_size=0.5):
    """ Preprocess the point cloud for registration """
    print(f"Original Point Cloud has {len(pcd.points)} points.")

    # Downsample using voxel grid filter
    print(f"Downsampling with voxel size {voxel_size}...")
    pcd_down = pcd.voxel_down_sample(voxel_size=voxel_size)
    print(f"Downsampled point cloud has {len(pcd_down.points)} points")

    # Estimate normals if they don't exist
    if not pcd_down.has_normals():
        print("Estimating normals...")
        pcd_down.estimate_normals(
            search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size*3, max_nn=50)  # Increased radius from 2 to 3 and max_nn from 30 to 50 for better normal estimation
        )

    # Remove outliers
    print("Removing outliers...")
    # Increased nb_neighbors from 20 to 30 and reduced std_ratio from 2.0 to 1.8 for more aggressive outlier removal
    cl, ind = pcd_down.remove_statistical_outlier(nb_neighbors=30, std_ratio=1.8)
    pcd_clean = pcd_down.select_by_index(ind)
    print(f"After outlier removal: {len(pcd_clean.points)} points")

    return pcd_clean

print('The Preprocessing has been finished.')


The Preprocessing has been finished.


### Using different downsampling size for preprocessing for Coarse(RANSAC) and Fine(ICP)

In [89]:
# ==== Defining downsampling for Coarse(RANSAC) and Fine(IPC) ====

# Create a copy of the original source point cloud for later use in ICP.
source_original = copy.deepcopy(source)

# Set voxel size for coarse registration (RANSAC):
# Using large voxel size (0.5-1.0m) for coarse registration as per supervisor's recommendation.
# Reduced from 0.60 to 0.40 to capture more details while still being efficient
ransac_voxel_size = 0.40 # Large voxel size for RANSAC (0.4 meters).


# ====  Adjusting downsampling for Coarse(RANSAC) ====

# Preprocess source and target point clouds for Coarse Registration(RANSAC).
print(f"Preprocessing source point cloud for Coarse registration(RANSAC) with large voxel size....")
source_down_coarse = preprocess_point_cloud(source, ransac_voxel_size)

print(f"\nPreprocessing target point cloud for Coarse registration(RANSAC) with large voxel size....")
target_down_coarse = preprocess_point_cloud(target, ransac_voxel_size)

# ====  Adjusting downsampling for Fine Registration(ICP) ====

# Create finely downsampled versions for ICP.
# Using fine Registration voxel size (0.05-0.1m) for ICP as per supervisor's recommendation.
# Reduced from 0.20 to 0.10 to capture more details for better fine registration
icp_voxel_size = 0.10  # Fine Registration(ICP) voxel size for ICP (10 cm)

print("\nCreating fine(ICP) downsampled source point cloud for ICP...")
source_down_fine = preprocess_point_cloud(source, icp_voxel_size)

print("\nCreating fine(ICP) downsampled target point cloud for ICP...")
target_down_fine = preprocess_point_cloud(target, icp_voxel_size)

Preprocessing source point cloud for Coarse registration(RANSAC) with large voxel size....
Original Point Cloud has 7182404 points.
Downsampling with voxel size 0.4...
Downsampled point cloud has 130994 points
Removing outliers...
After outlier removal: 128450 points

Preprocessing target point cloud for Coarse registration(RANSAC) with large voxel size....
Original Point Cloud has 20451227 points.
Downsampling with voxel size 0.4...
Downsampled point cloud has 181873 points
Estimating normals...
Removing outliers...
After outlier removal: 176583 points

Creating fine(ICP) downsampled source point cloud for ICP...
Original Point Cloud has 7182404 points.
Downsampling with voxel size 0.1...
Downsampled point cloud has 1475031 points
Removing outliers...
After outlier removal: 1437424 points

Creating fine(ICP) downsampled target point cloud for ICP...
Original Point Cloud has 20451227 points.
Downsampling with voxel size 0.1...
Downsampled point cloud has 2085412 points
Estimating norma

## 5. Feature-based Initial Alignment

Before applying ICP for fine registration, we'll perform an initial alignment using feature matching. This step is crucial for ICP to converge to the correct solution, especially when the point clouds are far apart initially.

We'll use FPFH (Fast Point Feature Histograms) features for this purpose.

In [90]:
def compute_fpfh_features(pcd, voxel_size):
    """ Compute FPFH features for the point cloud """
    radius_normal = voxel_size * 4  # Increased from 3 to 4 to capture more context for normal estimation
    radius_features = voxel_size * 10  # Increased from 8 to 10 to capture more context for feature computation

    # Ensure normals are computed.
    if not pcd.has_normals():
        pcd.estimate_normals(
            search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=70)  # Increased from 50 to 70 for better normal estimation
        )
    # Compute FPFH Features.
    print("Computing FPFH features...")
    fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd, 
        o3d.geometry.KDTreeSearchParamHybrid(radius=radius_features, max_nn=200)  # Increased max_nn from 150 to 200 for more comprehensive features
    )
    return fpfh

# Compute FPFH features for coarse Registration(RANSAC) downsampled source and target.

# For the Source.
print("Computing features for coarsely downsampled source point cloud...")
source_fpfh = compute_fpfh_features(source_down_coarse, ransac_voxel_size)

# For the Target.
print("\nComputing features for coarsely downsampled target point cloud...")
target_fpfh = compute_fpfh_features(target_down_coarse, ransac_voxel_size)

Computing features for coarsely downsampled source point cloud...
Computing FPFH features...

Computing features for coarsely downsampled target point cloud...
Computing FPFH features...


### Global Coarse Registration using RANSAC

In [91]:
# Perform global registration using RANSAC with coarsely downsampled point clouds.
print(f"\nPerforming global registration using RANSAC with coarsely downsampled point clouds...")
distance_threshold = ransac_voxel_size * 2.5  # Reduced from 3.5 to 2.5 for more precise matching with smaller voxel size
result_ransac = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(

    source_down_coarse, target_down_coarse, source_fpfh, target_fpfh, True, distance_threshold,  # Changed mutual_filter to True for more robust matching
    o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
    4, # Increased minimum number of correspondences from 3 to 4 for more robust transformation estimation

    [
        o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),  # Adjusted from 0.95 to 0.9 for better balance
        o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(distance_threshold * 2.0)  # Adjusted distance threshold for checker
    ], o3d.pipelines.registration.RANSACConvergenceCriteria(20000000, 0.9999)  # Increased to 20M iterations with 0.9999 confidence for better convergence

)


# Print Coarse Registration result
print("\nGlobal Coarse Registration(RANSAC) result:")
print(f"Fitness of Coarse Registration(RANSAC): {result_ransac.fitness}")
print(f"Inlier RMSE of Coarse Registration(RANSAC): {result_ransac.inlier_rmse}")
print("Transformation matrix of Coarse Registration(RANSAC):")
print(result_ransac.transformation)


Performing global registration using RANSAC with coarsely downsampled point clouds...
[Open3D WARNING] Too few correspondences (281) after mutual filter, fall back to original correspondences.

Global Coarse Registration(RANSAC) result:
Fitness of Coarse Registration(RANSAC): 0.19936940443752432
Inlier RMSE of Coarse Registration(RANSAC): 0.5836615166213526
Transformation matrix of Coarse Registration(RANSAC):
[[ 4.63293250e-01  5.72024087e-01 -6.76866167e-01  2.16151549e+02]
 [-7.88399215e-01  6.14837746e-01 -2.00305986e-02 -2.87880084e+02]
 [ 4.04704883e-01  5.42920796e-01  7.35833518e-01  1.37079694e+02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


## 6. Understanding Registration Results

Before visualizing the results, let's understand what the registration metrics mean:

- **Fitness**: Represents the overlap between point clouds after registration. It ranges from 0 to 1, where:
  - 0 means no overlap (poor registration)
  - 1 means perfect overlap (excellent registration)
  - Values above 0.3-0.4 typically indicate good registration

- **Inlier RMSE (Root Mean Square Error)**: Measures the average distance between corresponding points. Lower values indicate better alignment:
  - Values close to the voxel size or smaller indicate good alignment
  - High values suggest poor alignment

- **Transformation Matrix**: A 4x4 matrix representing the transformation:
  - An identity matrix (all 1's on diagonal, 0's elsewhere) means no transformation was found
  - The upper-left 3x3 submatrix represents rotation
  - The rightmost column represents translation
  - In a good result, you should see non-zero values in the rotation and/or translation components

If you see an identity matrix with zero fitness, it means the registration algorithm couldn't find a valid transformation. This can happen when:
1. The point clouds are too different in structure
2. The initial positions are too far apart
3. There aren't enough distinctive features for matching
4. The algorithm parameters need adjustment

## 7. Fine Registration using ICP

Now that we have a good initial alignment, we'll refine it using the ICP (Iterative Closest Point) algorithm. We'll try both point-to-point and point-to-plane ICP to see which gives better results.

**Apply the initial transformation to a copy of the source point cloud**

**This is needed for the ICP step**:
- source_initial = copy.deepcopy(source_down_coarse)
- source_initial.transform(result_ransac.transformation)

In [ ]:
# Function to perform ICP registration
def perform_icp(source, target, initial_transform, method="point_to_point", max_distance=0.15):
    """Perform ICP registration with the specified method"""
    if method == "point_to_point":
        print("\nPerforming point-to-point ICP...")
        estimation = o3d.pipelines.registration.TransformationEstimationPointToPoint()
    else:  # point_to_plane
        print("\nPerforming point-to-plane ICP...")
        estimation = o3d.pipelines.registration.TransformationEstimationPointToPlane()

    # Ensure both point clouds have normals for point-to-plane ICP
    if method == "point_to_plane":
        if not source.has_normals():
            source.estimate_normals(
                search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=icp_voxel_size*3, max_nn=50)
            )
        if not target.has_normals():
            target.estimate_normals(
                search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=icp_voxel_size*3, max_nn=50)
            )

    # Perform ICP
    result_icp = o3d.pipelines.registration.registration_icp(
        source, target, max_distance, initial_transform,
        estimation,
        o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=500, relative_fitness=1e-8, relative_rmse=1e-8)
    )

    return result_icp

# MODIFIED: Perform point-to-point ICP using finely downsampled point clouds
# Using fine voxel size (10 cm) for ICP as per supervisor's recommendation
print("\nPerforming point-to-point ICP with finely downsampled point clouds...")
result_icp_p2p = perform_icp(
    source_down_fine, target_down_fine, 
    result_ransac.transformation, 
    method="point_to_point", 
    max_distance=icp_voxel_size * 1.5  # Reduced from 2 to 1.5 times voxel size for more precise matching
)

# Print point-to-point ICP result
print("\nPoint-to-point ICP registration result:")
print(f"Fitness: {result_icp_p2p.fitness}")
print(f"Inlier RMSE: {result_icp_p2p.inlier_rmse}")
print("Transformation matrix:")
print(result_icp_p2p.transformation)

# MODIFIED: Perform point-to-plane ICP using finely downsampled point clouds
# Using fine voxel size (10 cm) for ICP as per supervisor's recommendation
print("\nPerforming point-to-plane ICP with finely downsampled point clouds...")
result_icp_p2l = perform_icp(
    source_down_fine, target_down_fine, 
    result_ransac.transformation, 
    method="point_to_plane", 
    max_distance=icp_voxel_size * 1.5  # Reduced from 2 to 1.5 times voxel size for more precise matching
)

# Print point-to-plane ICP result
print("\nPoint-to-plane ICP registration result:")
print(f"Fitness: {result_icp_p2l.fitness}")
print(f"Inlier RMSE: {result_icp_p2l.inlier_rmse}")
print("Transformation matrix:")
print(result_icp_p2l.transformation)


Performing point-to-point ICP with finely downsampled point clouds...

Performing point-to-point ICP...


## 8. Choose the Best Registration Result

Let's compare the results of point-to-point and point-to-plane ICP and choose the better one based on fitness and RMSE.

In [ ]:
if result_icp_p2p.fitness > result_icp_p2l.fitness:
    print("Point-to-point ICP gave better results.")
    final_result = result_icp_p2p
    method_name = "point-to-point"
else:
    print("Point-to-plane ICP gave better results.")
    final_result = result_icp_p2l
    method_name = "point-to-plane"

print(f"\nFinal registration result ({method_name} ICP):")
print(f"Fitness: {final_result.fitness}")
print(f"Inlier RMSE: {final_result.inlier_rmse}")

# Calculate the combined transformation (initial alignment + ICP refinement)
combined_transformation = final_result.transformation


# Apply the final transformation to the point clouds
# Transform the finely downsampled source point cloud using the final transformation
# Using the finely downsampled version (5 cm) for evaluation as per supervisor's recommendation
source_down_fine_aligned = copy.deepcopy(source_down_fine)
source_down_fine_aligned.transform(combined_transformation)

# Also transform the original source point cloud for saving
# This is done to maintain the full detail of the original point cloud as per supervisor's recommendation
source_original_aligned = copy.deepcopy(source_original)
source_original_aligned.transform(combined_transformation)

## 10. Save the Aligned Map

Finally, let's save the aligned map for future use.

In [ ]:
# Save the aligned original source point cloud
# Using the original point cloud for saving to maintain full detail as per supervisor's recommendation
aligned_file = "Final_map_Registration.ply"
o3d.io.write_point_cloud(aligned_file, source_original_aligned)
print(f"Aligned map saved to: {aligned_file}")
print("Note: The saved map is the original (full resolution) version as per supervisor's recommendation.")

# Save the transformation matrix for reference
np.savetxt("Final_map_transformation_matrix.txt", combined_transformation)
print("Transformation matrix saved to: transformation_matrix.txt")

## 12. Conclusion

In this notebook, we have successfully performed point cloud registration to align the map generated by KISS-ICP with the reference labeled map. The registration process involved:

1. **Loading the original point cloud** (Final_Kiss_SLAM_Map_Original_Cropped.ply) as per supervisor's recommendation
2. **Multi-resolution preprocessing** with different voxel sizes:
   - Large voxel size (0.5 meters) for coarse registration (RANSAC)
   - Fine voxel size (5 cm) for fine registration (ICP)
3. **Initial alignment** using feature-based methods (FPFH features with RANSAC) on coarsely downsampled point clouds
4. **Fine registration** using ICP (both point-to-point and point-to-plane variants) on finely downsampled point clouds
5. **Evaluation** of the registration quality using finely downsampled point clouds
6. **Saving** the aligned original (full resolution) map for future use

### Optimization Strategy

Following the supervisor's recommendations, we implemented a multi-resolution approach:

1. Used the original point cloud as the source for the entire pipeline
2. Applied coarse downsampling (0.5 meters) for the computationally intensive RANSAC step
3. Applied fine downsampling (5 cm) for the ICP step to maintain accuracy
4. Transformed and saved the original point cloud to preserve all details

This approach balances computational efficiency with registration accuracy:
- Coarse downsampling reduces computation time for the initial alignment
- Fine downsampling maintains sufficient detail for accurate ICP registration
- Using the original point cloud for the final result preserves all details

The downsampled aligned map has been saved as `aligned_map.ply` and the transformation matrix as `transformation_matrix.txt`.